# MineSense — Fine-tuning LLM (QLoRA) di Google Colab / Kaggle (GPU T4 gratis)
Runtime → Change runtime type → **T4 GPU**. Estimasi total ±45–60 menit untuk Qwen2.5-1.5B, 2 epoch.

Alur: setup → data sintetis → dataset SFT → QLoRA → evaluasi → merge → GGUF → (opsional) push ke Hugging Face Hub.

In [ ]:
# 1) Ambil kode: clone dari GitHub Anda (ganti URL) ATAU upload minesense.zip ke /content
import os
REPO_URL = "https://github.com/<username>/minesense.git"  # ganti
if not os.path.exists("/content/minesense"):
    if os.path.exists("/content/minesense.zip"):
        !cd /content && unzip -q minesense.zip
    else:
        !git clone $REPO_URL /content/minesense
%cd /content/minesense
!pip install -q -e ".[train]"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Data sintetis + dataset SFT (ganti data/*.csv dengan data asli bila sudah ada)
!python -m minesense.data.synth
!python -m minesense.finetune.build_sft_dataset

In [ ]:
# 3) Training QLoRA
!python -m minesense.finetune.train_lora --base Qwen/Qwen2.5-1.5B-Instruct --epochs 2 --run-name qwen15b-v1

In [ ]:
# 4) Uji cepat model hasil fine-tune (adapter) dengan transformers
import json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
ad = "artifacts/lora/qwen15b-v1/adapter"
tok = AutoTokenizer.from_pretrained(ad)
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, ad).eval()
row = json.loads(open("data/sft/val.jsonl").readline())
msgs = row["messages"][:2]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
out = model.generate(ids, max_new_tokens=300, do_sample=False)
print("TASK:", row["task"]); print("PRED:", tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)); print("GOLD:", row["messages"][2]["content"])

In [ ]:
# 5) Merge + konversi GGUF (untuk Ollama di CPU/VPS)
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF > /dev/null && cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null
!python -m minesense.finetune.merge_and_export --adapter artifacts/lora/qwen15b-v1/adapter --name minesense-qwen15b --gguf q4_k_m --llama-cpp /content/llama.cpp
!ls -lh artifacts/gguf

In [ ]:
# 6) (Opsional) Simpan ke Hugging Face Hub / Google Drive
# from huggingface_hub import login; login()
# !huggingface-cli upload <username>/minesense-qwen15b-gguf artifacts/gguf
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/minesense && cp -r artifacts/gguf artifacts/lora /content/drive/MyDrive/minesense/

## 7) Deploy model hasil fine-tune
Di server/VPS: salin `minesense-qwen15b-q4_k_m.gguf` + `Modelfile` ke folder `models/`, lalu:
```bash
docker compose --profile ollama up -d
docker compose exec ollama ollama create minesense -f /models/Modelfile
# .env: MINESENSE_LLM_PROVIDER=ollama  MINESENSE_LLM_MODEL=minesense
python -m minesense.finetune.eval_llm   # bandingkan dengan base model
```